In [ ]:
import xarray as xr 
import torch
from dssml.utils.nbplot import plot_vars_two_rows

In [ ]:
from dssml.data.splitters import BlockWindowSplitter
from dssml.data.modules.meps import ZarrWindowDataModule
from dssml.data.normalizers import SymRangeNormalizer

In [ ]:
zarr_path = "/ec/res4/hpcperm/smcd/data/aifs-meps-2.5km-2020-2023-1h-v2_SMHI_subdomain_reduced.zarr" # Atos 
zarr_path = "/ec/res4/hpcperm/smcd/data/aifs-meps-2.5km-2020-2023-1h-v2_SMHI_subdomain_reduced_time_7_chunked.zarr" # Atos 



In [ ]:
normalizer = SymRangeNormalizer()

In [ ]:
wsplit = BlockWindowSplitter(train=100, val=1, test=1, skip=1, window_size=3)

In [ ]:
normalizer =  SymRangeNormalizer()

In [ ]:
dm = ZarrWindowDataModule(dataset_path = zarr_path, num_workers=2, normalizer = normalizer, splitter = wsplit, pin_memory=False, prefetch_factor=4 , batch_size=32, cells_reshape = (256,256) , variables=["10u", "2t"])

In [ ]:
dm.setup("fit")     # build train/val/test datasets


In [ ]:
train_ds = dm.train_ds
len(train_ds)

In [ ]:
print(train_ds[0].shape)

In [ ]:
for i in range(0,10):
    print(train_ds[i].shape)


In [ ]:
import time

train_loader = dm.train_dataloader()

# This is the most efficient way to loop manually
for batch_idx, batch in enumerate(train_loader):
    # The loader is already working on batch_idx + 1 in the background
    # because of num_workers and prefetch_factor.
    
    # Process your batch
    print(f"Batch {batch_idx} shape: {batch.shape}")
    
    if batch_idx > 5:
        break
print("finished")

In [ ]:
batch = next(iter(train_loader))
batch.shape


In [ ]:
train_ds = dm.train_ds
print("Train samples:", len(train_ds))

# One item (dataset returns (T, V, C) if drop_ensemble=True)
x = train_ds[0]
print("\nSingle sample:")
print("  x shape:", tuple(x.shape), "| dtype:", x.dtype)  # expect (T, V, C)

# ---------- Sanity stats on the sample ----------
x_f = x.float()

print("\nSample Global Stats:")
print(
    f"  Min: {x_f.min().item():.4f}, "
    f"Max: {x_f.max().item():.4f}, "
    f"Mean: {x_f.mean().item():.4f}, "
    f"Std: {x_f.std().item():.4f}"
)

# Per-variable stats over time+cell (dims 0 and 2)
print("\nSample Per-variable Stats (reduce over time+cell):")
print(f"{'Var':<4} | {'Min':<10} | {'Max':<10} | {'Mean':<10} | {'Std':<10}")
print("-" * 60)

num_vars = x.shape[1]
for v in range(num_vars):
    v_data = x_f[:, v, :].reshape(-1)
    print(
        f"{v:<4} | "
        f"{v_data.min().item():<10.4f} | "
        f"{v_data.max().item():<10.4f} | "
        f"{v_data.mean().item():<10.4f} | "
        f"{v_data.std().item():<10.4f}"
    )

# ---------- Verify loaded stats match the intended variables ----------
print("\nLoaded dm.stats:")
for k, t in dm.stats.items():
    if t is None:
        print(f"  {k}: None")
    else:
        print(f"  {k}: shape={tuple(t.shape)}, dtype={t.dtype}")

# Stats should be per-variable vectors and match x.shape[1]
V = x.shape[1]
assert dm.stats["vmin"] is not None and dm.stats["vmax"] is not None, "dm.stats must contain vmin/vmax"
assert dm.stats["vmin"].numel() == V and dm.stats["vmax"].numel() == V, (
    f"Stats length mismatch: sample has V={V}, stats have "
    f"vmin={dm.stats['vmin'].numel()} vmax={dm.stats['vmax'].numel()}"
)

# Compare stats values to the sample (note: sample mins/maxes won't equal dataset-global mins/maxes)
print("\nStats sanity check:")
print("  Variables:", getattr(dm, "selected_variables", None))
print("  dm.stats[vmin] (first 5):", dm.stats["vmin"][: min(5, V)].tolist())
print("  dm.stats[vmax] (first 5):", dm.stats["vmax"][: min(5, V)].tolist())
if dm.stats.get("mean") is not None:
    print("  dm.stats[mean] (first 5):", dm.stats["mean"][: min(5, V)].tolist())
if dm.stats.get("std") is not None:
    print("  dm.stats[std]  (first 5):", dm.stats["std"][: min(5, V)].tolist())


In [ ]:
x = train_ds[10]  # (T, V, 256, 256)
plot_vars_two_rows(x, t_idx=0, titles=getattr(dm, "selected_variables", None))